# NIH45-46: S54G and I51L attenuate the L45P penalty

Matched pairs of NIH45-46 genotypes differing only at L45P, restricted to N52K-mutant backgrounds, grouped by which of S54G and I51L are present. Each group is compared against the no-modifier group by Welch's t-test on the per-pair deltas, corrected across the three comparisons by Benjamini-Hochberg.

## 1. Setup

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests

## 2. Paths and constants

In [ ]:
# ---- paths -------------------------------------------------------------
data_dir = "../data"
table_dir = "../tables"
os.makedirs(table_dir, exist_ok=True)

# display name to file stem; the hyphens and the dot do not appear in any filename
NAME_TO_STEM = {"1NC9": "1NC9", "CH235.12": "CH235_12", "N49P7": "N49P7",
                "NIH45-46": "NIH4546", "VRC-PG20": "VRCPG20", "VRC01": "VRC01",
                "VRC07": "VRC07", "VRC13": "VRC13", "VRC16": "VRC16"}


GERMLINE = "G"                  # site_seq character meaning "germline here"
VALUE_COL = "nlog10_Kd_raw"     # raw (not Tobit-imputed) affinity column

ANTIBODY = "NIH45-46"
FOCAL, COND = "L45P", "N52K"
MODIFIERS = ["S54G", "I51L"]


## 3. Data loaders

In [ ]:
def load_raw_binding_data():
    """Pooled genotype table assembled from the per-antibody kds files.

    One row per antibody, antigen and genotype, replicate-averaged, with the
    G/M token string in site_seq. Reconstructs what the old pooled
    final_cleaned_all_antibody_binding_data.csv provided.
    """
    frames = []
    for ab in NAME_TO_STEM:
        d = pd.read_csv(f"{data_dir}/{NAME_TO_STEM[ab]}_kds.csv")
        d["antibody"] = ab
        d["site_seq"] = d[[f"site_{i}" for i in range(10)]].agg("".join, axis=1)
        frames.append(d)
    df = pd.concat(frames, ignore_index=True)
    df = (df.groupby(["antibody", "antigen", "site_seq", "id"], as_index=False)["nlog10_Kd"]
            .mean()
            .rename(columns={"nlog10_Kd": "nlog10_Kd_raw"}))
    print(f"Loaded raw binding data: {df.shape} from {len(NAME_TO_STEM)} kds files")
    return df


def kabat_to_site_index(antibody):
    # {'L45P': 3, ...} -- maps a mutation label to its index in site_seq.
    meta = pd.read_csv(f"{data_dir}/{NAME_TO_STEM[antibody]}_meta.csv")
    meta.columns = meta.columns.str.strip()
    return {f"{r['germline_aa']}{r['kabat_position']}{r['mature_aa']}":
            int(str(r["site"]).replace("site_", "")) for _, r in meta.iterrows()}

## 4. Matched-pair construction (inlined)

For each background (the genotype string with the focal site removed), the
germline and mature sides are averaged separately and then joined, so one row
= one exactly-matched background. `cont` records the contingent mutation's
state (here N52K) and `rem` records which of the tracked modifiers are present.

In [ ]:
def build_split_frame(df, antibody, antigen, focal_mut, contingent_mut, remaining_muts, k2i):
    f_idx = k2i[focal_mut]
    cont_idx = k2i[contingent_mut]
    rem_idx = [k2i[m] for m in remaining_muts]

    d = df[(df.antibody == antibody) & (df.antigen == antigen)].dropna(subset=[VALUE_COL]).copy()
    d["_f"] = d["site_seq"].str[f_idx] != GERMLINE

    def bgkey(seq):
        return seq[:f_idx] + seq[f_idx + 1:]
    d["_bg"] = d["site_seq"].apply(bgkey)
    d["_cont"] = d["site_seq"].str[cont_idx] != GERMLINE
    d["_rem"] = d["site_seq"].apply(lambda s: tuple(i for i in rem_idx if s[i] != GERMLINE))

    germ = d[~d["_f"]].groupby("_bg").agg(germ_val=(VALUE_COL, "mean"), cont=("_cont", "first"), rem=("_rem", "first"))
    mat = d[d["_f"]].groupby("_bg")[VALUE_COL].mean().rename("mat_val")
    merged = germ.join(mat, how="inner")
    merged["delta"] = merged["mat_val"] - merged["germ_val"]
    return merged, rem_idx

## 5. The table

In [ ]:
def modifier_delta_table(df, antigen="SF162", cond_state=True):
    """cond_state=True restricts to N52K-MUTANT backgrounds (False = germline)."""
    k2i = kabat_to_site_index(ANTIBODY)
    d = df[(df.antibody == ANTIBODY) & (df.antigen == antigen)].dropna(subset=[VALUE_COL]).copy()

    merged, rem_idx = build_split_frame(d, ANTIBODY, antigen, FOCAL, COND, MODIFIERS, k2i)
    merged = merged[merged["cont"] == cond_state].copy()

    s54g_idx, i51l_idx = k2i["S54G"], k2i["I51L"]
    merged["delta"] = merged["mat_val"] - merged["germ_val"]
    merged["has_S54G"] = merged["rem"].apply(lambda r: s54g_idx in r)
    merged["has_I51L"] = merged["rem"].apply(lambda r: i51l_idx in r)

    groups = {
        "Neither":   merged[~merged.has_S54G & ~merged.has_I51L],
        "S54G only": merged[merged.has_S54G & ~merged.has_I51L],
        "I51L only": merged[~merged.has_S54G & merged.has_I51L],
        "S54G+I51L": merged[merged.has_S54G & merged.has_I51L],
    }
    base = groups["Neither"]["delta"].to_numpy()

    rows = []
    for name, g in groups.items():
        vals = g["delta"].to_numpy()
        row = {"background": name, "n": len(vals), "mean_delta": vals.mean()}
        if name == "Neither":
            row.update({"shift_vs_neither": np.nan, "p_raw": np.nan})
        else:
            row["shift_vs_neither"] = vals.mean() - base.mean()
            row["p_raw"] = ttest_ind(vals, base, equal_var=False).pvalue
        rows.append(row)

    tab = pd.DataFrame(rows)
    # BH-FDR across the three modifier-vs-neither comparisons only.
    mask = tab["p_raw"].notna()
    tab.loc[mask, "p_bh"] = multipletests(tab.loc[mask, "p_raw"], method="fdr_bh")[1]
    tab["significance"] = tab["p_bh"].apply(
        lambda p: "--" if pd.isna(p) else
                  "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "n.s.")
    return tab

## 6. Run

In [ ]:
df = load_raw_binding_data()

tab_sf162 = modifier_delta_table(df, antigen="SF162")
tab_sf162.to_csv(f"{table_dir}/NIH4546_L45P_modifier_delta_table_SF162.csv", index=False)
print("NIH45-46 L45P delta by S54G/I51L -- SF162, N52K-mutant")
tab_sf162.round(4)

In [ ]:
# CH505TF, same design -- the antigen comparison.
tab_ch505 = modifier_delta_table(df, antigen="CH505TF")
tab_ch505.to_csv(f"{table_dir}/NIH4546_L45P_modifier_delta_table_CH505TF.csv", index=False)
tab_ch505.round(4)